# Baseline Model Results

Fit and evaluate the two reference baselines that the Bayesian model should do better than.
1) team-independent historical frequency model
2) the per-team Poisson model

In [1]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import pandas as pd
import numpy as np

from src.config import PROCESSED_DATA_DIR, TABLES_DIR, TRAIN_SEASONS, TEST_SEASONS
from src.baselines import HistoricalFrequencyBaseline, TeamPoissonBaseline

In [2]:
df = pd.read_csv(PROCESSED_DATA_DIR / "epl_matches.csv", parse_dates=["Date"])

train_df = df[df["season"].isin(TRAIN_SEASONS)].reset_index(drop=True)
test_df = df[df["season"].isin(TEST_SEASONS)].reset_index(drop=True)

print(f"Train: {len(train_df)} matches\nTest: {len(test_df)} matches")

Train: 2660 matches
Test: 380 matches


In [3]:
freq_model = HistoricalFrequencyBaseline().fit(train_df)
freq_preds = freq_model.predict(test_df)

print(f"H={freq_model.p_home_win:.3f}  D={freq_model.p_draw:.3f}  A={freq_model.p_away_win:.3f}")

freq_preds[["HomeTeam", "AwayTeam", "FTR", "p_home_win", "p_draw", "p_away_win"]].head()

H=0.442  D=0.235  A=0.323


,HomeTeam,AwayTeam,FTR,p_home_win,p_draw,p_away_win
0,Burnley,Man City,A,0.442481,0.234586,0.322932
1,Arsenal,Nott'm Forest,H,0.442481,0.234586,0.322932
2,Bournemouth,West Ham,D,0.442481,0.234586,0.322932
3,Brighton,Luton,H,0.442481,0.234586,0.322932
4,Everton,Fulham,A,0.442481,0.234586,0.322932


In [4]:
poisson_model = TeamPoissonBaseline().fit(train_df)
poisson_preds = poisson_model.predict(test_df)

poisson_preds[["HomeTeam", "AwayTeam", "FTR", "p_home_win", "p_draw", "p_away_win"]].head()

,HomeTeam,AwayTeam,FTR,p_home_win,p_draw,p_away_win
0,Burnley,Man City,A,0.093574,0.181120,0.725306
1,Arsenal,Nott'm Forest,H,0.612179,0.213599,0.174223
2,Bournemouth,West Ham,D,0.336110,0.234077,0.429814
3,Brighton,Luton,H,0.309699,0.273802,0.416499
4,Everton,Fulham,A,0.643586,0.211844,0.144570


In [5]:
for label, preds in [("Frequency", freq_preds), ("Poisson", poisson_preds)]:
    total = preds[["p_home_win", "p_draw", "p_away_win"]].sum(axis=1)

    pred_label = preds[["p_home_win", "p_draw", "p_away_win"]].idxmax(axis=1).map(
        {"p_home_win": "H", "p_draw": "D", "p_away_win": "A"}
    )
    acc = (pred_label == preds["FTR"]).mean()
    print(f"{label}-based Baseline Model: accuracy {acc:.3f}")

Frequency-based Baseline Model: accuracy 0.461
Poisson-based Baseline Model: accuracy 0.513


In [7]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)

freq_preds.to_csv(TABLES_DIR / "baseline_freq_preds.csv", index=False)
poisson_preds.to_csv(TABLES_DIR / "baseline_poisson_preds.csv", index=False)